In [ ]:
#!pip install torchtext

from torchtext.data import Field, LabelField
from torchtext.datasets import IMDB

TEXT = Field(sequential=True, lower=True, batch_first=True)
LABEL = LabelField(dtype=None)

# Load the dataset by passing the fields
train_dataset, test_dataset = IMDB.splits(TEXT, LABEL)


### preprocessing steps before feeding the dataset to rnn:
1. Split the training dataset into separate training and validation partitions.
2. Identify the unique words in the training dataset
3. Map each unique word to a unique integer and encode the review text into encoded integers (an index of each unique word)
4. Divide the dataset into mini-batches as input to the model

In [ ]:
#1st step: creating the dataset for training and validation
from torch.utils.data.dataset import random_split
import torch
torch.manual_seed(1)

# Convert to a list to get its stable size
train_list = list(train_dataset)
total_len = len(train_list)
train_len = int(0.8 * total_len)
valid_len = total_len - train_len

# Perform split using dynamic lengths
train_split, valid_split = random_split(train_list, [train_len, valid_len])

# finding uniques codes 2nd step

import re
import collections
from collections import Counter, OrderedDict

def tokenizer(text):
  text = re.sub('<[^>]>', '', text)
  emoticoins = re.findall(r'(?::|;| = )(?:-)?(?:\)|"|\(|D|P)', text.lower())
  text = re.sub(r'[\W] + ', ' ', text.lower()) + ' '.join(emoticoins).replace('-', '')
  tokenized = text.split()
  return tokenized

tokens_counts = Counter()
for example in train_split:
  tokens = tokenizer(" ".join(example.text))
  tokens_counts.update(tokens)
print('vocab-size: ', len(tokens_counts))
print(f'total len {total_len}')

In [ ]:
# 3rd step: encoding each token to tokens through vocab
# how Vocab works in torchtext 0.6.0:
from torchtext.vocab import Vocab

# Pass the Counter object directly, and define special tokens in the constructor
vocab = Vocab(tokens_counts, specials=['<pad>', '<unk>'])

# In torchtext 0.6.0, <unk> is at index 0 by default. 
# Let's verify how vocab works on sample tokens:
# Use .stoi (string-to-index) to look up indices, defaulting to the <unk> index (0) if not found
print("Index of '<pad>':", vocab.stoi['<pad>'])
print("Index of '<unk>':", vocab.stoi['<unk>'])
print("Index of 'this':", vocab.stoi['this'])

# How to safely look up tokens with a fallback for unknown words:
unk_idx = vocab.stoi['<unk>']
sample_indices = [vocab.stoi.get(token, unk_idx) for token in ['this', 'is', 'a', 'example']]
print("Encoded sample:", sample_indices)

In [ ]:
# defining function of tokenizer
import torch
import torch.nn as nn

text_pipeline = lambda x: [vocab[token] for token in tokenizer(x)]
label_pipeline = lambda x: 1 if x == 'pos' else 0

# wrap the encode and transformation batch :
def collate_batch(batch):
  lengths, label_list, text_list = [], [], []

  for _label, _text in batch:
    label_list.append(label_pipeline(_label))
    processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
    text_list.append(processed_text)
    lengths.append(processed_text.size(0))

  # These must be computed OUTSIDE the for loop
  padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=vocab.stoi['<pad>'])
  lengths = torch.tensor(lengths)
  label_list = torch.tensor(label_list)

  return padded_text_list, lengths, label_list

In [ ]:
# We pass the standard Python list of tuples to the DataLoader
from torch.utils.data import DataLoader
dataloader = DataLoader(train_data_list, batch_size=4, shuffle=False, collate_fn=collate_batch)

text_batch, lengths_batch, label_batch = next(iter(dataloader))
print("Text batch shape:", text_batch.shape)
print("Lengths:", lengths_batch)
print("Labels:", label_batch)

batch_size = 32
train_split, valid_split = random_split(train_list, [train_len, valid_len])
train_dl = DataLoader(train_split, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_dl = DataLoader(valid_split, batch_size=batch_size,  shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

# Embedding layers for sentence encoding